In [1]:
import pandas as pd
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn

In [2]:
# Load the CSV file
df_train = pd.read_csv('mnist_train_small.csv')
df_test = pd.read_csv('mnist_test.csv')

In [3]:
print(df_train.columns)

Index(['6', '0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8',
       ...
       '0.581', '0.582', '0.583', '0.584', '0.585', '0.586', '0.587', '0.588',
       '0.589', '0.590'],
      dtype='object', length=785)


In [4]:
# Extract features and labels
X_train = df_train.drop('6', axis=1).values
y_train = df_train['6'].values
print(y_train[1000:1500])

[9 1 0 3 0 2 0 7 7 9 9 7 8 8 8 2 7 6 7 6 0 0 8 9 3 7 1 7 0 9 5 6 5 1 2 8 8
 1 2 0 9 0 9 3 0 0 7 5 7 8 5 5 6 6 5 6 1 3 6 9 9 1 1 8 3 7 5 9 4 0 3 4 0 1
 7 1 3 5 6 3 4 7 0 4 1 9 3 2 8 0 4 8 4 0 1 1 9 3 3 1 5 2 4 0 1 5 6 0 4 5 0
 7 9 6 9 6 9 0 5 5 2 2 2 7 4 5 2 9 7 3 2 4 8 4 2 4 0 3 9 6 8 9 6 9 2 5 0 0
 9 9 4 2 5 6 6 1 1 1 6 7 3 2 1 5 2 0 2 6 0 4 6 1 8 6 6 1 1 4 0 0 9 5 1 9 1
 3 6 5 9 9 9 6 6 6 8 1 4 7 0 1 8 0 0 4 7 0 5 7 7 4 7 3 5 1 6 3 8 0 6 2 8 6
 8 2 6 2 5 5 1 9 9 8 8 2 9 3 8 7 7 9 5 0 0 2 2 6 3 1 2 1 7 4 5 7 0 6 4 5 2
 8 3 8 2 3 9 1 8 7 3 2 6 4 2 3 8 0 7 4 5 1 5 6 2 5 4 8 7 7 2 6 0 5 2 4 3 1
 5 4 0 8 9 1 2 8 4 4 7 9 4 3 3 7 1 4 8 7 6 4 7 8 0 1 7 7 7 1 4 7 5 7 5 4 4
 4 6 1 1 9 2 1 2 8 3 9 7 1 9 4 3 6 1 4 9 3 7 7 1 0 6 9 0 0 4 9 2 2 4 8 3 1
 4 1 6 2 7 0 3 2 4 7 6 2 2 2 3 9 4 0 2 4 6 7 6 4 8 2 1 2 9 1 4 1 1 3 7 2 8
 6 5 3 6 8 1 6 6 7 8 9 4 0 5 2 9 7 2 1 0 0 0 4 6 9 4 9 8 6 2 3 9 7 6 9 3 8
 2 1 8 4 1 2 4 9 9 5 8 8 7 1 5 6 6 6 4 8 1 8 5 5 7 3 3 9 4 4 7 4 3 4 1 0 4
 1 3 1 3 3 6 6 5 1 5 0 5 

In [5]:
print(df_test.columns)

Index(['7', '0', '0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8',
       ...
       '0.658', '0.659', '0.660', '0.661', '0.662', '0.663', '0.664', '0.665',
       '0.666', '0.667'],
      dtype='object', length=785)


In [6]:
X_test = df_test.drop('7', axis=1).values
y_test = df_test['7'].values
print(y_test[1000:1500])

[0 2 5 1 9 7 8 1 0 4 1 7 9 6 4 2 6 8 1 3 7 5 4 4 1 8 1 3 8 1 2 5 8 0 6 2 1
 1 7 1 5 3 4 6 9 5 0 9 2 2 4 8 2 1 7 2 4 9 4 4 0 3 9 2 2 3 3 8 3 5 7 3 5 8
 1 2 4 4 6 4 9 5 1 0 6 9 5 9 5 9 7 3 8 0 3 7 1 3 6 7 8 5 9 7 9 6 9 6 3 7 4
 4 5 3 5 4 7 8 7 8 0 7 6 8 8 7 3 3 1 9 5 2 7 3 5 1 1 2 1 4 7 4 7 5 4 5 4 0
 8 3 6 9 6 0 2 7 4 4 4 4 6 6 4 7 9 3 4 5 5 8 7 3 7 2 7 0 2 4 1 1 6 6 9 2 8
 7 2 0 1 5 0 9 1 7 0 6 0 8 6 8 1 8 0 3 3 7 2 3 6 2 1 6 1 1 3 7 9 0 8 0 5 4
 0 2 8 7 2 9 8 4 0 9 5 8 5 1 2 1 3 1 7 4 5 7 2 0 9 8 8 6 2 5 4 1 9 2 1 5 8
 7 0 2 4 4 3 6 8 8 2 4 0 5 0 4 4 7 9 3 4 1 5 9 7 3 5 8 8 0 5 3 3 6 6 0 1 6
 0 3 5 4 4 1 2 9 1 4 6 9 9 3 9 8 4 4 3 1 3 1 8 8 7 9 4 8 8 7 9 7 1 4 5 6 0
 5 2 2 2 1 5 5 2 4 9 6 2 7 7 2 2 1 1 2 8 3 7 2 4 1 7 1 7 6 7 8 2 7 3 1 7 5
 8 2 6 2 2 5 6 5 0 9 2 4 3 3 9 7 6 6 8 0 4 1 5 8 2 9 1 8 0 6 7 2 1 0 5 5 2
 0 2 2 0 2 4 9 8 0 9 9 4 6 5 4 9 1 8 3 4 9 9 1 2 2 8 1 9 6 4 0 9 4 8 3 8 6
 0 2 5 1 9 6 2 9 4 0 9 6 0 6 2 5 4 2 3 8 4 5 5 0 3 8 5 3 5 8 6 5 7 6 3 3 9
 6 1 1 2 9 0 4 3 3 6 9 5 

In [7]:
# Feature scaling
def scale(x):
    scal = (x-x.mean(axis=0))/(x.max(axis=0)-x.min(axis=0))
    return np.nan_to_num(scal)

# Scale features
x_train = scale(X_train)
x_test = scale(X_test)

C:\Users\TUF\AppData\Local\Temp\ipykernel_17824\419983731.py:3: RuntimeWarning: invalid value encountered in divide
  scal = (x-x.mean(axis=0))/(x.max(axis=0)-x.min(axis=0))


In [8]:
import torch.nn.functional as F

# Convert to PyTorch tensors
tx_train = torch.tensor(x_train, dtype=torch.float32)
ty_train = torch.tensor(y_train, dtype=torch.long)
tx_test = torch.tensor(x_test, dtype=torch.float32)
ty_test = torch.tensor(y_test, dtype=torch.long)
m_train = tx_train.size(0)
m_test = tx_test.size(0)

# Model
def model(tx_train, th1, b1, th2, b2, th3, b3):
    # Hidden layer 1
    out = torch.matmul(tx_train, th1) + b1
    out = torch.sigmoid(out)

    # Hidden layer 2
    out = torch.matmul(out, th2) + b2
    out = torch.sigmoid(out)

    # Output layer
    out = torch.matmul(out, th3) + b3
    return out

In [9]:
print(torch.unique(ty_train))

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [10]:
# Initialize parameters
th1 = torch.rand(784, 28, requires_grad=True, dtype=torch.float32)
th2 = torch.rand(28, 12, requires_grad=True, dtype=torch.float32)
th3 = torch.rand(12, 10, requires_grad=True, dtype=torch.float32)  
b1 = torch.rand(1, 28, requires_grad=True, dtype=torch.float32)
b2 = torch.rand(1, 12, requires_grad=True, dtype=torch.float32)
b3 = torch.rand(1, 10, requires_grad=True, dtype=torch.float32)  

In [11]:
# Optimizer
opt = optim.SGD([th1, th2, th3, b1, b2, b3], lr=1)
cost_func = nn.CrossEntropyLoss()

In [12]:
# Training
for iter in range(1, 10000):
    out = model(tx_train, th1, b1, th2, b2, th3, b3)

    # Cost function
    cost = cost_func(out, ty_train)
    cost.backward()

    if iter % 40 == 0:
        print(f'iter: {iter:02d} => cost: {cost.item():.4f}')

    # Update parameters
    opt.step()
    opt.zero_grad()

iter: 40 => cost: 2.2355
iter: 80 => cost: 2.1874
iter: 120 => cost: 2.1646
iter: 160 => cost: 2.1560
iter: 200 => cost: 2.1520
iter: 240 => cost: 2.1493
iter: 280 => cost: 2.1470
iter: 320 => cost: 2.1446
iter: 360 => cost: 2.1419
iter: 400 => cost: 2.1385
iter: 440 => cost: 2.1342
iter: 480 => cost: 2.1284
iter: 520 => cost: 2.1200
iter: 560 => cost: 2.1072
iter: 600 => cost: 2.0859
iter: 640 => cost: 2.0608
iter: 680 => cost: 2.0334
iter: 720 => cost: 2.0013
iter: 760 => cost: 1.9657
iter: 800 => cost: 1.9275
iter: 840 => cost: 1.8870
iter: 880 => cost: 1.8447
iter: 920 => cost: 1.8007
iter: 960 => cost: 1.7576
iter: 1000 => cost: 1.7186
iter: 1040 => cost: 1.6810
iter: 1080 => cost: 1.6336
iter: 1120 => cost: 1.5853
iter: 1160 => cost: 1.5335
iter: 1200 => cost: 1.4827
iter: 1240 => cost: 1.4370
iter: 1280 => cost: 1.3944
iter: 1320 => cost: 1.3554
iter: 1360 => cost: 1.3198
iter: 1400 => cost: 1.2858
iter: 1440 => cost: 1.2530
iter: 1480 => cost: 1.2236
iter: 1520 => cost: 1.1957


In [13]:
# Evaluate on training set
out = model(tx_train, th1, b1, th2, b2, th3, b3) # tensor(m, 3)
predict = torch.argmax(out, dim=1) # tensor(m)
accuracy = (predict==ty_train).sum()/m_train
print(accuracy.item())

0.9621981382369995


In [14]:
# Convert to PyTorch tensors
tx_test = torch.tensor(x_test, dtype=torch.float32) # tensor(10000, 784)
ty_test = torch.tensor(y_test, dtype=torch.long) # tensor(10000, )
m_test = tx_test.size(0)

In [16]:
# Evaluate on test set
out = model(tx_test, th1, b1, th2, b2, th3, b3) # tensor(m_test, 3)
predict = torch.argmax(out, dim=1) # tensor(m_test)
accuracy = (predict==ty_test).sum()/m_test
print(accuracy.item())

0.9978997707366943


In [17]:
# Initialize parameters
th1 = torch.rand(784, 256, requires_grad=True, dtype=torch.float32)
th2 = torch.rand(256, 64, requires_grad=True, dtype=torch.float32)
th3 = torch.rand(64, 32, requires_grad=True, dtype=torch.float32)
th4 = torch.rand(32, 16, requires_grad=True, dtype=torch.float32)
th5 = torch.rand(16, 10, requires_grad=True, dtype=torch.float32)

b1 = torch.rand(1, 256, requires_grad=True, dtype=torch.float32)
b2 = torch.rand(1, 64, requires_grad=True, dtype=torch.float32)
b3 = torch.rand(1, 32, requires_grad=True, dtype=torch.float32)
b4 = torch.rand(1, 16, requires_grad=True, dtype=torch.float32)
b5 = torch.rand(1, 10, requires_grad=True, dtype=torch.float32)

In [18]:
# 5 layers model
# Model
def model5(tx, th1, b1, th2, b2, th3, b3, th4, b4, th5, b5):
    # Hidden layer 1
    out = torch.matmul(tx, th1) + b1
    out = torch.sigmoid(out)

    # Hidden layer 2
    out = torch.matmul(out, th2) + b2
    out = torch.sigmoid(out)

    # Hidden layer 3
    out = torch.matmul(out, th3) + b3
    out = torch.sigmoid(out)

        # Hidden layer 4
    out = torch.matmul(out, th4) + b4
    out = torch.sigmoid(out)

    # Output layer
    out = torch.matmul(out, th5) + b5
    return out

print(tx_train.shape)

torch.Size([19999, 784])


In [19]:
print(torch.unique(ty_train))

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [20]:
# Define optimizer
learning_rate5 = 0.001
optimizer = optim.Adam([th1, b1, th2, b2, th3, b3, th4, b4, th5, b5], lr=learning_rate5)

# Training loop
for iter in range(1, 10000):
    out = model5(tx_train, th1, b1, th2, b2, th3, b3, th4, b4, th5, b5)

    # Cost function
    cost = cost_func(out, ty_train)
    cost.backward()

    if iter % 100 == 0:
        print(f'iter: {iter:02d} => cost: {cost.item():.4f}')

    # Update parameters
    optimizer.step()
    optimizer.zero_grad()

iter: 100 => cost: 2.3717
iter: 200 => cost: 2.3011
iter: 300 => cost: 2.3007
iter: 400 => cost: 2.3007
iter: 500 => cost: 2.3007
iter: 600 => cost: 2.3007
iter: 700 => cost: 2.3007
iter: 800 => cost: 2.3007
iter: 900 => cost: 2.3007
iter: 1000 => cost: 2.3007
iter: 1100 => cost: 2.3007
iter: 1200 => cost: 2.3007
iter: 1300 => cost: 2.3007
iter: 1400 => cost: 2.3007
iter: 1500 => cost: 2.3007
iter: 1600 => cost: 2.3007
iter: 1700 => cost: 2.3007
iter: 1800 => cost: 2.3007
iter: 1900 => cost: 2.3007
iter: 2000 => cost: 2.3007
iter: 2100 => cost: 2.3007
iter: 2200 => cost: 2.3007
iter: 2300 => cost: 2.3007
iter: 2400 => cost: 2.3007
iter: 2500 => cost: 2.3007
iter: 2600 => cost: 2.3007
iter: 2700 => cost: 2.3007
iter: 2800 => cost: 2.3007
iter: 2900 => cost: 2.3007
iter: 3000 => cost: 2.3007
iter: 3100 => cost: 2.3007
iter: 3200 => cost: 2.3007
iter: 3300 => cost: 2.3007
iter: 3400 => cost: 2.3007
iter: 3500 => cost: 2.3007
iter: 3600 => cost: 2.3007
iter: 3700 => cost: 2.3007
iter: 3800

In [21]:
# Evaluate on training set
out = model5(tx_train, th1, b1, th2, b2, th3, b3, th4, b4, th5, b5) # tensor(m, 5)
predict = torch.argmax(out, dim=1) # tensor(m)
accuracy = (predict==ty_train).sum()/m_train
print(accuracy.item())

0.11215560883283615


In [23]:
# Evaluate on test set
out = model5(tx_test, th1, b1, th2, b2, th3, b3, th4, b4, th5, b5) # tensor(m_test, 3)
predict = torch.argmax(out, dim=1) # tensor(m_test)
accuracy = (predict==ty_test).sum()/m_test
print(accuracy.item())

0.1135113537311554


In [24]:
class NeuralNetworkRelu(nn.Module):
    def __init__(self):
        super(NeuralNetworkRelu, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn1 = nn.BatchNorm1d(256)  # Batch normalization layer
        self.fc2 = nn.Linear(256, 64)
        self.bn2 = nn.BatchNorm1d(64)  # Batch normalization layer
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        return x

# Initialize parameters
model2 = NeuralNetworkRelu()
optimizer = optim.SGD(model2.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(3000):
    optimizer.zero_grad()
    output = model2(tx_train.view(-1, 28*28))
    loss = criterion(output, ty_train)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.item()}')

Epoch: 0, Loss: 2.3738811016082764
Epoch: 100, Loss: 0.18085481226444244
Epoch: 200, Loss: 0.08206777274608612
Epoch: 300, Loss: 0.043600715696811676
Epoch: 400, Loss: 0.026126425713300705
Epoch: 500, Loss: 0.017339251935482025
Epoch: 600, Loss: 0.012462236918509007
Epoch: 700, Loss: 0.009502606466412544
Epoch: 800, Loss: 0.0075727831572294235
Epoch: 900, Loss: 0.006240237969905138
Epoch: 1000, Loss: 0.005269122775644064
Epoch: 1100, Loss: 0.004539041314274073
Epoch: 1200, Loss: 0.0039712307043373585
Epoch: 1300, Loss: 0.0035196575336158276
Epoch: 1400, Loss: 0.003153212834149599
Epoch: 1500, Loss: 0.002850637771189213
Epoch: 1600, Loss: 0.0025972495786845684
Epoch: 1700, Loss: 0.002382187405601144
Epoch: 1800, Loss: 0.002197584602981806
Epoch: 1900, Loss: 0.0020376923494040966
Epoch: 2000, Loss: 0.0018979088636115193
Epoch: 2100, Loss: 0.0017748272512108088
Epoch: 2200, Loss: 0.0016656470252200961
Epoch: 2300, Loss: 0.0015682006487622857
Epoch: 2400, Loss: 0.0014807459665462375
Epoch:

In [25]:
# Define the evaluate function
def evaluate(model, tx, ty):
    with torch.no_grad():
        output = model(tx)
        loss = criterion(output, ty)
        predict = torch.argmax(output, dim=1)
        accuracy = (predict == ty).sum().item() / ty.size(0)
    return loss.item(), accuracy

In [26]:
# Evaluate on training set
train_loss, train_accuracy = evaluate(model2, tx_train, ty_train)
print(f"Training Loss: {train_loss}, Training Accuracy: {train_accuracy}")

Training Loss: 0.001101578352972865, Training Accuracy: 1.0


In [28]:
# Evaluate on test set
test_loss, test_accuracy = evaluate(model2, tx_test, ty_test)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

Test Loss: 0.00037421929300762713, Test Accuracy: 1.0


In [29]:
# Model definition with ReLU activation for hidden layers and softmax for output
class NeuralNetworkSoftmax(nn.Module):
    def __init__(self):
        super(NeuralNetworkSoftmax, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.bn1 = nn.BatchNorm1d(256)  # Batch normalization layer
        self.fc2 = nn.Linear(256, 64)
        self.bn2 = nn.BatchNorm1d(64)  # Batch normalization layer
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.fc3(x)
        x = self.softmax(x)
        return x

# Initialize parameters
model3 = NeuralNetworkSoftmax()
optimizer = optim.SGD(model3.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(3000):
    optimizer.zero_grad()
    output = model3(tx_train)
    loss = criterion(output, ty_train)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.item()}')

Epoch: 0, Loss: 2.304842948913574
Epoch: 100, Loss: 1.7560018301010132
Epoch: 200, Loss: 1.6298030614852905
Epoch: 300, Loss: 1.5546499490737915
Epoch: 400, Loss: 1.5269699096679688
Epoch: 500, Loss: 1.5102996826171875
Epoch: 600, Loss: 1.4994755983352661
Epoch: 700, Loss: 1.4919977188110352
Epoch: 800, Loss: 1.4865325689315796
Epoch: 900, Loss: 1.4824384450912476
Epoch: 1000, Loss: 1.4792895317077637
Epoch: 1100, Loss: 1.4768767356872559
Epoch: 1200, Loss: 1.4750699996948242
Epoch: 1300, Loss: 1.473524570465088
Epoch: 1400, Loss: 1.4722682237625122
Epoch: 1500, Loss: 1.4712841510772705
Epoch: 1600, Loss: 1.470467209815979
Epoch: 1700, Loss: 1.4698055982589722
Epoch: 1800, Loss: 1.4692305326461792
Epoch: 1900, Loss: 1.4687315225601196
Epoch: 2000, Loss: 1.46833074092865
Epoch: 2100, Loss: 1.4679628610610962
Epoch: 2200, Loss: 1.4676076173782349
Epoch: 2300, Loss: 1.4673314094543457
Epoch: 2400, Loss: 1.4670836925506592
Epoch: 2500, Loss: 1.4668407440185547
Epoch: 2600, Loss: 1.46657896

In [30]:
# Evaluate on training set
train_loss, train_accuracy = evaluate(model3, tx_train, ty_train)
print(f"Training Loss: {train_loss}, Training Accuracy: {train_accuracy}")

Training Loss: 1.465667486190796, Training Accuracy: 0.9970498524926247


In [32]:
# Evaluate on test set
test_loss, test_accuracy = evaluate(model3, tx_test, ty_test)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

Test Loss: 1.4626359939575195, Test Accuracy: 0.9988998899889989
